# Kinetic isotope effects with PyQuiver

A **kinetic isotope effect (KIE)** is the ratio of reaction rates for two isotopologues, molecules that differ only by an isotopic substitution (for example ¹²C vs ¹³C at one atom). A heavier isotope lowers vibrational frequencies, and hence zero-point energies; because that change differs between the reactant and the transition state, the free-energy barrier, and so the rate, shifts slightly. The electronic barrier itself is unchanged, so a KIE is a sensitive, geometry-specific probe of mechanism.

PyQuiver computes KIEs from the vibrational frequencies of the reactant (the *ground state*) and the *transition state* using the Bigeleisen-Mayer equation. You provide the Cartesian Hessian from a frequency calculation on each, plus a short configuration of the substitutions to make.

In [1]:
import pandas as pd
import cctk                       # optional; only used to read energies for Skodje-Truhlar
from IPython.display import display

from pyquiver import KIE_Calculation, Config, System, Isotopologue, batch
from pyquiver.constants import DEFAULT_MASSES, REPLACEMENTS
from pyquiver.kie import calculate_rpfr, partition_components

## Your first calculation

A calculation needs a **configuration**, a **ground-state** file, and a **transition-state** file. Build the configuration in Python with `Config.from_dict`: each isotopologue is a name mapped to a list of `(ground_atom, ts_atom, isotope)` substitutions, with atoms numbered as in your output file. Here we place a single ¹³C at atom 1. (`temperature` is in K, `scaling` is an empirical frequency-scaling factor, and `imag_threshold` in cm⁻¹ is the cutoff below which a small imaginary frequency is ignored rather than treated as the reaction mode.)

In [2]:
config = Config.from_dict(
    isotopologues={"C1": [(1, 1, "13C")]},
    temperature=393, scaling=0.9614, imag_threshold=50,
)
calc = KIE_Calculation(config, "../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out",
                       style="gaussian")
print(calc)


=== PyQuiver Analysis ===
Isotopologue                                              uncorrected      Wigner     inverted parabola
                                                              KIE           KIE              KIE
Isotopologue         C1                                      1.0127        1.0144          1.0147      

Absolute KIEs are given.


## Reading the result

Each isotopologue gets three numbers: the **uncorrected** KIE and two tunnelling-corrected versions (**Wigner** and **Bell**, explained below). A value above 1 is a *normal* KIE (the light isotope reacts faster); below 1 is *inverse*. The ¹³C effect at C1 here is a small normal KIE near 1.01, typical of a carbon whose bonding changes at the transition state.

In [3]:
print(calc.results["C1"])        # one isotopologue, by name
display(calc.to_dataframe())     # the whole set as a table

KIEResult(name='C1', uncorrected=1.012746371067466, wigner=1.014443004304243, inverted_parabola=1.014743121186644)


,name,uncorrected,wigner,inverted_parabola
0,C1,1.012746,1.014443,1.014743


## Substituting several positions

List more than one tuple to substitute several atoms in the same isotopologue, and use any `weights.dat` label (`13C`, `2D`, `17O`, ...); a bare number works too, for an unusual mass. Below, `H/D` deuterates two hydrogens at once. Note its KIE is *inverse* (below 1), an inverse secondary deuterium effect, while ¹³C and ¹⁷O give small normal effects.

In [4]:
config = Config.from_dict(
    isotopologues={
        "C1":  [(1, 1, "13C")],
        "O3":  [(3, 3, "17O")],
        "H/D": [(7, 7, "2D"), (8, 8, "2D")],   # two atoms at once
    },
    temperature=393, scaling=0.9614, imag_threshold=50,
)
calc = KIE_Calculation(config, "../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out",
                       style="gaussian")
display(calc.to_dataframe())

,name,uncorrected,wigner,inverted_parabola
0,C1,1.012746,1.014443,1.014743
1,O3,1.018842,1.020365,1.020635
2,H/D,0.954837,0.956311,0.956572


## Other input formats

The same configuration runs on Gaussian, ORCA (`.hess`), or PyQuiver's native `.qin` files; just change `style`.

In [5]:
orca   = KIE_Calculation(config, "../tutorial/orca/claisen_gs_freq.hess", "../tutorial/orca/claisen_ts_freq.hess", style="orca")
native = KIE_Calculation(config, "../tutorial/pyquiver/claisen_gs.qin", "../tutorial/pyquiver/claisen_ts.qin", style="pyquiver")

print(f"Gaussian C1: {calc.results['C1'].uncorrected:.4f}")
print(f"ORCA     C1: {orca.results['C1'].uncorrected:.4f}")
print(f"native   C1: {native.results['C1'].uncorrected:.4f}")

Gaussian C1: 1.0127
ORCA     C1: 1.0130
native   C1: 1.0127


## Parameter scans

Reading a file into a `System` parses it once, so you can reuse it across many calculations, for instance to scan temperature:

In [6]:
gs = System("../tutorial/gaussian/claisen_gs.out")
ts = System("../tutorial/gaussian/claisen_ts.out")

for T in (298, 350, 393):
    cfg = Config.from_dict(isotopologues={"C1": [(1, 1, "13C")]},
                           temperature=T, scaling=0.9614, imag_threshold=50)
    print(f"{T} K: C1 KIE = {KIE_Calculation(cfg, gs, ts).results['C1'].uncorrected:.4f}")

298 K: C1 KIE = 1.0144
350 K: C1 KIE = 1.0134
393 K: C1 KIE = 1.0127


## Many structures at once

`batch` runs one configuration over many ground-state/transition-state pairs and returns a single table. Give it a `{label: (gs, ts)}` dictionary (or a list of `(label, gs, ts)` triples, or bare `(gs, ts)` pairs labelled by filename).

In [7]:
results = batch(config, {
    "b3lyp": ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
    "m06":   ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
})
display(results.to_dataframe())

,label,name,uncorrected,wigner,inverted_parabola
0,b3lyp,C1,1.012746,1.014443,1.014743
1,b3lyp,O3,1.018842,1.020365,1.020635
2,b3lyp,H/D,0.954837,0.956311,0.956572
3,m06,C1,1.012746,1.014443,1.014743
4,m06,O3,1.018842,1.020365,1.020635
5,m06,H/D,0.954837,0.956311,0.956572


## Tunnelling corrections

Light nuclei can tunnel through the barrier near the transition state, speeding the reaction up. PyQuiver always reports the **uncorrected** KIE plus two corrections, **Wigner** and **Bell** (the inverted parabola); Bell is the default. For heavy-atom KIEs (¹³C, ¹⁷O) the corrections barely change the number, so the choice rarely matters. For hydrogen they can be large, and for a **primary** H/D KIE (the deuterated atom lies in the reaction coordinate) the harmonic treatment is only approximate, so PyQuiver warns and the prediction should be treated with caution.

Writing $u = \frac{h c\, \tilde\nu}{k_B T}$ for an imaginary mode of magnitude $\tilde\nu$ (cm⁻¹), the single-mode coefficients are

$$\kappa_{\text{Wigner}} = 1 + \frac{u^2}{24}, \qquad \kappa_{\text{Bell}} = \frac{u/2}{\sin(u/2)},$$

and the factor applied to a KIE is $\kappa_{\text{light}}/\kappa_{\text{heavy}}$.

### Skodje-Truhlar

When the imaginary frequency is large the inverted parabola is unreliable, and the Skodje-Truhlar correction (*J. Phys. Chem.* **1981**, *85*, 624) is preferable. It needs the barrier height, so you supply the reactant, transition-state, and product energies.

The natural frequency of the imaginary mode (from its force constant $k^{\ddagger}$ and reduced mass $\mu^{\ddagger}$), handled automatically by PyQuiver, is

$$\nu^{\ddagger} = \frac{1}{2\pi}\sqrt{\frac{k^{\ddagger}}{\mu^{\ddagger}}}.$$

With $\alpha = \frac{2\pi}{h\nu^{\ddagger}}$, $\beta = \frac{1}{k_B T}$ (so $\nu^{\ddagger} = c\,\tilde\nu^{\ddagger}$ is in Hz), and the barrier height in the exothermic direction $V = V_{\text{TS}} - \max(V_{\text{reactant}}, V_{\text{product}})$, there are three cases.

For a small $\nu^{\ddagger}$ / high $T$ ($\alpha > \beta$):

$$\kappa = \frac{\beta\pi/\alpha}{\sin(\beta\pi/\alpha)} - \frac{\beta}{\alpha-\beta}\,e^{(\beta-\alpha)V}.$$

For a large $\nu^{\ddagger}$ / low $T$ ($\alpha < \beta$):

$$\kappa = \frac{\beta}{\beta-\alpha}\left(e^{(\beta-\alpha)V} - 1\right).$$

And at the crossover $\alpha = \beta$:

$$\kappa = \frac{\beta\pi/\alpha}{\sin(\beta\pi/\alpha)}.$$

Below the crossover temperature ($\alpha < \beta$) the correction grows without bound, so PyQuiver warns there. As with Wigner and Bell, the factor applied to a KIE is $\kappa_{\text{light}}/\kappa_{\text{heavy}}$.

In [8]:
def energy(path):
    ensemble = cctk.GaussianFile.read_file(path).ensemble
    return ensemble.get_properties_dict(ensemble.molecules[-1])["energy"]   # hartree

E_reactant = energy("../tutorial/gaussian/claisen_gs.out")
E_ts = energy("../tutorial/gaussian/claisen_ts.out")

# product taken equal to the reactant here
calc.skodje_truhlar(reactant_energy=E_reactant, product_energy=E_reactant, ts_energy=E_ts)

OrderedDict([('C1', np.float64(1.0147431211866438)),
             ('O3', np.float64(1.0206350979560215)),
             ('H/D', np.float64(0.956572091397035))])

## Under the hood: partition function ratios

*(Optional.)* A KIE is the ratio of two reduced isotopic partition function ratios (RPFRs), one for the ground state and one for the transition state, times the classical imaginary-frequency ratio and the tunnelling factor. You can compute an RPFR directly from a light and a heavy `Isotopologue`:

In [9]:
gs_masses = [DEFAULT_MASSES[z] for z in gs.atomic_numbers]
heavy_masses = list(gs_masses)
heavy_masses[0] = REPLACEMENTS["13C"]        # 13C at atom 1

light = Isotopologue("light", gs, gs_masses)
heavy = Isotopologue("heavy", gs, heavy_masses)

rpfr, imag_ratio, heavy_freqs, light_freqs = calculate_rpfr((light, heavy), 50.0, 0.9614, 393)
print(f"ground-state RPFR = {rpfr:.4f}")

ground-state RPFR = 1.0872


Each mode's contribution to the RPFR factors into a frequency ratio, an excitation term, and a zero-point term (their product is the contribution):

In [10]:
components = partition_components(heavy_freqs, light_freqs, 393)
modes = pd.DataFrame(components, columns=["freq_ratio", "excitation", "ZPE"])
modes.insert(0, "light_freq", light_freqs)
modes.insert(1, "heavy_freq", heavy_freqs)
modes["contribution"] = modes[["freq_ratio", "excitation", "ZPE"]].prod(axis=1)
display(modes.head())

,light_freq,heavy_freq,freq_ratio,excitation,ZPE,contribution
0,67.790652,67.225890,0.991669,1.007402,1.001034,1.000043
1,159.689475,159.684161,0.999967,1.000024,1.000010,1.000001
2,179.501554,178.499592,0.994418,1.003970,1.001836,1.000199
3,246.525660,246.070898,0.998155,1.001138,1.000833,1.000123
4,276.476300,273.262636,0.988376,1.006802,1.005900,1.000971
